# PathVQA Data Preprocessing

This notebook implements data preprocessing for the PathVQA dataset according to Section 3.2 of the assessment document.

## Preprocessing Steps:
1. **Question Type Classification**: Classify questions into closed-ended vs open-ended
2. **Image Preprocessing**: Resize, crop, and normalize images
3. **Text Preprocessing**: Normalize and tokenize questions and answers

## Dataset:
- Source: `flaviagiammarino/path-vqa` from HuggingFace
- Hardware: Gradient Paperspace Free-A6000 GPU


In [1]:
# Install required packages
%pip install -q datasets transformers pillow torch torchvision


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
import os
import re
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset, DatasetDict
from torchvision import transforms
from transformers import AutoTokenizer
from collections import Counter
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


c:\Users\xianz\anaconda3\envs\ml_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.5.1
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
# Configuration
DATASET_NAME = "flaviagiammarino/path-vqa"
OUTPUT_DIR = "preprocessed_path_vqa"
IMAGE_SIZE = 224  # Can be changed to 384 if needed
MAX_QUESTION_LENGTH = 128
MAX_ANSWER_LENGTH = 64
TOKENIZER_NAME = "google/flan-t5-base"  # Or can use CLIP tokenizer later

print(f"Configuration:")
print(f"  Dataset: {DATASET_NAME}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Max question length: {MAX_QUESTION_LENGTH}")
print(f"  Max answer length: {MAX_ANSWER_LENGTH}")
print(f"  Tokenizer: {TOKENIZER_NAME}")


Configuration:
  Dataset: flaviagiammarino/path-vqa
  Output directory: preprocessed_path_vqa
  Image size: 224x224
  Max question length: 128
  Max answer length: 64
  Tokenizer: google/flan-t5-base


In [4]:
# Load dataset from HuggingFace
print("Loading dataset from HuggingFace...")
dataset = load_dataset(DATASET_NAME)
print(f"\nDataset structure: {dataset}")
print(f"Split names: {list(dataset.keys())}")

# Inspect first example
print("\n=== First Example from Train Set ===")
first_example = dataset['train'][0]
print(f"Keys: {first_example.keys()}")
for key, value in first_example.items():
    if key == 'image':
        print(f"  {key}: PIL Image, size={value.size}, mode={value.mode}")
    else:
        print(f"  {key}: {type(value).__name__} = {str(value)[:100]}")


Loading dataset from HuggingFace...

Dataset structure: DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 19654
    })
    validation: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6259
    })
    test: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6719
    })
})
Split names: ['train', 'validation', 'test']

=== First Example from Train Set ===
Keys: dict_keys(['image', 'question', 'answer'])
  image: PIL Image, size=(309, 272), mode=CMYK
  question: str = where are liver stem cells (oval cells) located?
  answer: str = in the canals of hering


In [5]:
# Question Type Classification Functions
def classify_question_type(question):
    """
    Classify question as closed-ended or open-ended based on question word.
    Closed-ended: Yes/No questions
    Open-ended: What, Where, How, Why, When, etc.
    """
    question_lower = question.lower().strip()
    
    # Closed-ended question indicators
    closed_indicators = ['is', 'are', 'was', 'were', 'do', 'does', 'did', 
                        'can', 'could', 'will', 'would', 'has', 'have', 'had']
    
    # Check if starts with closed-ended question word
    for indicator in closed_indicators:
        if question_lower.startswith(indicator + ' '):
            return 'closed-ended'
    
    # Open-ended question indicators
    open_indicators = ['what', 'where', 'how', 'why', 'when', 'which', 'who', 'whose', 
                       'how many', 'how much']
    
    # Check for open-ended question words
    for indicator in open_indicators:
        if question_lower.startswith(indicator):
            return 'open-ended'
    
    # Default to open-ended if unclear
    return 'open-ended'

# Test the classification function
test_questions = [
    "Is this a malignant tumor?",
    "What type of tissue is shown?",
    "Where is the lesion located?",
    "Are there any abnormalities?",
    "How many cells are visible?",
    "Why is this significant?"
]

print("=== Question Type Classification Test ===")
for q in test_questions:
    q_type = classify_question_type(q)
    print(f"  [{q_type:15s}] {q}")


=== Question Type Classification Test ===
  [closed-ended   ] Is this a malignant tumor?
  [open-ended     ] What type of tissue is shown?
  [open-ended     ] Where is the lesion located?
  [closed-ended   ] Are there any abnormalities?
  [open-ended     ] How many cells are visible?
  [open-ended     ] Why is this significant?


In [6]:
# Analyze question types in the dataset
print("Analyzing question types in the dataset...")
question_type_counts = {'closed-ended': 0, 'open-ended': 0}
question_word_counts = Counter()

for split in ['train', 'validation', 'test']:
    if split in dataset:
        for example in tqdm(dataset[split], desc=f"Analyzing {split}"):
            question = example['question']
            q_type = classify_question_type(question)
            question_type_counts[q_type] += 1
            
            # Get question word
            first_word = question.split()[0].lower() if question else 'unknown'
            question_word_counts[first_word] += 1

print("\n=== Question Type Distribution ===")
total = sum(question_type_counts.values())
for q_type, count in question_type_counts.items():
    percentage = (count / total) * 100
    print(f"  {q_type:15s}: {count:6d} ({percentage:5.2f}%)")

print("\n=== Top 10 Question Words ===")
for word, count in question_word_counts.most_common(10):
    print(f"  {word:15s}: {count:6d}")


Analyzing question types in the dataset...


Analyzing test: 100%|██████████| 6719/6719 [01:10<00:00, 95.78it/s] 


=== Question Type Distribution ===
  closed-ended   :  16235 (49.75%)
  open-ended     :  16397 (50.25%)

=== Top 10 Question Words ===
  what           :  13337
  is             :   8623
  does           :   6267
  where          :   2155
  are            :    822
  how            :    695
  do             :    373
  why            :    114
  did            :     59
  when           :     49


In [7]:
# Initialize tokenizer
print(f"Loading tokenizer: {TOKENIZER_NAME}")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Tokenizer pad token: {tokenizer.pad_token}")

# Set pad token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Set pad_token to eos_token: {tokenizer.pad_token}")


Loading tokenizer: google/flan-t5-base
Tokenizer vocab size: 32100
Tokenizer pad token: <pad>


In [8]:
# Image preprocessing transform
# Using ImageNet normalization statistics
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),  # Resize to square
    transforms.ToTensor(),  # Convert to tensor [0, 1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]    # ImageNet std
    )
])

# Test image preprocessing
print("Testing image preprocessing...")
test_image = dataset['train'][0]['image']
print(f"Original image: size={test_image.size}, mode={test_image.mode}")

# Ensure RGB mode
if test_image.mode != 'RGB':
    test_image = test_image.convert('RGB')
    print(f"Converted to RGB: size={test_image.size}, mode={test_image.mode}")

# Apply transform
test_tensor = image_transform(test_image)
print(f"Processed tensor: shape={test_tensor.shape}, dtype={test_tensor.dtype}")
print(f"  Min value: {test_tensor.min():.4f}, Max value: {test_tensor.max():.4f}")
print(f"  Mean: {test_tensor.mean():.4f}, Std: {test_tensor.std():.4f}")


Testing image preprocessing...
Original image: size=(309, 272), mode=CMYK
Converted to RGB: size=(309, 272), mode=RGB
Processed tensor: shape=torch.Size([3, 224, 224]), dtype=torch.float32
  Min value: -2.1008, Max value: 2.6400
  Mean: 1.8729, Std: 0.7314


In [9]:
# Text preprocessing functions
def preprocess_text(text):
    """
    Basic text normalization:
    - Strip whitespace
    - Normalize whitespace
    - Convert to lowercase (optional, can be handled by tokenizer)
    """
    if not isinstance(text, str):
        text = str(text)
    
    # Strip and normalize whitespace
    text = ' '.join(text.split())
    return text.strip()

# Test text preprocessing
test_texts = [
    "  Is this   a malignant   tumor?  ",
    "What type of tissue is shown?",
    "  Multiple   spaces   here  "
]

print("=== Text Preprocessing Test ===")
for text in test_texts:
    processed = preprocess_text(text)
    print(f"  Original: '{text}'")
    print(f"  Processed: '{processed}'")
    print()


=== Text Preprocessing Test ===
  Original: '  Is this   a malignant   tumor?  '
  Processed: 'Is this a malignant tumor?'

  Original: 'What type of tissue is shown?'
  Processed: 'What type of tissue is shown?'

  Original: '  Multiple   spaces   here  '
  Processed: 'Multiple spaces here'



In [10]:
# Main preprocessing function
def preprocess_example(example):
    """
    Preprocess a single example:
    1. Classify question type
    2. Preprocess image
    3. Preprocess and tokenize text
    """
    # 1. Question type classification
    question = example['question']
    question_type = classify_question_type(question)
    
    # 2. Preprocess question
    question_processed = preprocess_text(question)
    
    # 3. Preprocess answer
    answer = example['answer']
    answer_processed = preprocess_text(answer)
    
    # 4. Tokenize question
    question_tokens = tokenizer(
        question_processed,
        padding='max_length',
        truncation=True,
        max_length=MAX_QUESTION_LENGTH,
        return_tensors=None
    )
    
    # 5. Tokenize answer (for sequence-to-sequence models)
    answer_tokens = tokenizer(
        answer_processed,
        padding='max_length',
        truncation=True,
        max_length=MAX_ANSWER_LENGTH,
        return_tensors=None
    )
    
    # 6. Preprocess image
    image = example['image']
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    # Convert PIL to tensor and normalize
    image_tensor = image_transform(image)
    
    # Convert tensor to list for storage (HuggingFace datasets)
    image_array = image_tensor.numpy().tolist()
    
    return {
        'question': question_processed,
        'answer': answer_processed,
        'question_type': question_type,
        'input_ids': question_tokens['input_ids'],
        'attention_mask': question_tokens['attention_mask'],
        'answer_input_ids': answer_tokens['input_ids'],
        'answer_attention_mask': answer_tokens['attention_mask'],
        'pixel_values': image_array,  # Store as list of lists
        'image_size': IMAGE_SIZE
    }

# Test preprocessing on a few examples
print("=== Testing Preprocessing Function ===")
test_example = dataset['train'][0]
print(f"Original example keys: {test_example.keys()}")
print(f"Original question: {test_example['question']}")
print(f"Original answer: {test_example['answer']}")

processed = preprocess_example(test_example)
print(f"\nProcessed example keys: {processed.keys()}")
print(f"Question type: {processed['question_type']}")
print(f"Question: {processed['question']}")
print(f"Answer: {processed['answer']}")
print(f"Input IDs length: {len(processed['input_ids'])}")
print(f"Pixel values type: {type(processed['pixel_values'])}")
print(f"Pixel values shape: {len(processed['pixel_values'])} channels")
if len(processed['pixel_values']) > 0:
    print(f"  Channel 0 shape: {len(processed['pixel_values'][0])}")


=== Testing Preprocessing Function ===
Original example keys: dict_keys(['image', 'question', 'answer'])
Original question: where are liver stem cells (oval cells) located?
Original answer: in the canals of hering

Processed example keys: dict_keys(['question', 'answer', 'question_type', 'input_ids', 'attention_mask', 'answer_input_ids', 'answer_attention_mask', 'pixel_values', 'image_size'])
Question type: open-ended
Question: where are liver stem cells (oval cells) located?
Answer: in the canals of hering
Input IDs length: 128
Pixel values type: <class 'list'>
Pixel values shape: 3 channels
  Channel 0 shape: 224


In [11]:
# Preprocess all splits
print("Starting preprocessing of all splits...")
preprocessed_datasets = {}

for split_name in ['train', 'validation', 'test']:
    if split_name in dataset:
        print(f"\n{'='*60}")
        print(f"Preprocessing {split_name} split...")
        print(f"{'='*60}")
        
        split_data = dataset[split_name]
        print(f"Original size: {len(split_data)}")
        
        # Apply preprocessing with batching for efficiency
        # Use num_proc=None to disable multiprocessing and avoid serialization issues
        preprocessed_split = split_data.map(
            preprocess_example,
            batched=False,  # Process one at a time to handle images properly
            num_proc=None,  # Disable multiprocessing to avoid function serialization issues
            desc=f"Preprocessing {split_name}"
        )
        
        preprocessed_datasets[split_name] = preprocessed_split
        print(f"Preprocessed size: {len(preprocessed_split)}")
        
        # Show statistics
        print(f"\n{split_name} Statistics:")
        q_type_counts = Counter(preprocessed_split['question_type'])
        for q_type, count in q_type_counts.items():
            print(f"  {q_type}: {count}")
        
        # Show example
        print(f"\nExample from {split_name}:")
        example = preprocessed_split[0]
        print(f"  Question: {example['question']}")
        print(f"  Answer: {example['answer']}")
        print(f"  Type: {example['question_type']}")
        print(f"  Input IDs length: {len(example['input_ids'])}")
        print(f"  Pixel values channels: {len(example['pixel_values'])}")
    else:
        print(f"Warning: Split '{split_name}' not found in dataset")


Starting preprocessing of all splits...

Preprocessing train split...
Original size: 19654


Preprocessing train: 100%|██████████| 19654/19654 [38:51<00:00,  8.43 examples/s]   


Preprocessed size: 19654

train Statistics:
  open-ended: 9905
  closed-ended: 9749

Example from train:
  Question: where are liver stem cells (oval cells) located?
  Answer: in the canals of hering
  Type: open-ended
  Input IDs length: 128
  Pixel values channels: 3

Preprocessing validation split...
Original size: 6259


Preprocessing validation: 100%|██████████| 6259/6259 [10:25<00:00, 10.01 examples/s]  


Preprocessed size: 6259

validation Statistics:
  open-ended: 3134
  closed-ended: 3125

Example from validation:
  Question: what have lost their nuclei?
  Answer: neutrophils
  Type: open-ended
  Input IDs length: 128
  Pixel values channels: 3

Preprocessing test split...
Original size: 6719


Preprocessing test: 100%|██████████| 6719/6719 [13:30<00:00,  8.29 examples/s]   


Preprocessed size: 6719

test Statistics:
  open-ended: 3358
  closed-ended: 3361

Example from test:
  Question: what are positively charged, thus allowing the compaction of the negatively charged dna?
  Answer: the histone subunits
  Type: open-ended
  Input IDs length: 128
  Pixel values channels: 3


In [12]:
# Create DatasetDict and save
preprocessed_dataset_dict = DatasetDict(preprocessed_datasets)
print(f"\nPreprocessed dataset structure: {preprocessed_dataset_dict}")
print(f"Splits: {list(preprocessed_dataset_dict.keys())}")

# Save to disk
print(f"\nSaving preprocessed dataset to '{OUTPUT_DIR}'...")
preprocessed_dataset_dict.save_to_disk(OUTPUT_DIR)
print(f"Dataset saved successfully!")

# Verify saved dataset
print("\nVerifying saved dataset...")
verify_dataset = DatasetDict.load_from_disk(OUTPUT_DIR)
print(f"Verified dataset structure: {verify_dataset}")
print(f"Verified splits: {list(verify_dataset.keys())}")
print(f"Train size: {len(verify_dataset['train'])}")
print(f"Validation size: {len(verify_dataset['validation'])}")
print(f"Test size: {len(verify_dataset['test'])}")


Preprocessed dataset structure: DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer', 'question_type', 'input_ids', 'attention_mask', 'answer_input_ids', 'answer_attention_mask', 'pixel_values', 'image_size'],
        num_rows: 19654
    })
    validation: Dataset({
        features: ['image', 'question', 'answer', 'question_type', 'input_ids', 'attention_mask', 'answer_input_ids', 'answer_attention_mask', 'pixel_values', 'image_size'],
        num_rows: 6259
    })
    test: Dataset({
        features: ['image', 'question', 'answer', 'question_type', 'input_ids', 'attention_mask', 'answer_input_ids', 'answer_attention_mask', 'pixel_values', 'image_size'],
        num_rows: 6719
    })
})
Splits: ['train', 'validation', 'test']

Saving preprocessed dataset to 'preprocessed_path_vqa'...


Saving the dataset (3/17 shards):  18%|█▊        | 1107/6259 [00:10<00:48, 106.54 examples/s]


OSError: [Errno 28] No space left on device

In [ ]:
# Additional statistics and analysis
print("\n" + "="*60)
print("FINAL STATISTICS")
print("="*60)

for split_name in ['train', 'validation', 'test']:
    if split_name in preprocessed_dataset_dict:
        split_data = preprocessed_dataset_dict[split_name]
        
        print(f"\n{split_name.upper()} Split:")
        print(f"  Total examples: {len(split_data)}")
        
        # Question type distribution
        q_types = split_data['question_type']
        closed_count = sum(1 for qt in q_types if qt == 'closed-ended')
        open_count = sum(1 for qt in q_types if qt == 'open-ended')
        print(f"  Closed-ended: {closed_count} ({closed_count/len(split_data)*100:.2f}%)")
        print(f"  Open-ended: {open_count} ({open_count/len(split_data)*100:.2f}%)")
        
        # Answer statistics
        answers = split_data['answer']
        answer_lengths = [len(a.split()) for a in answers]
        print(f"  Answer length - Mean: {np.mean(answer_lengths):.2f}, "
              f"Median: {np.median(answer_lengths):.2f}, "
              f"Max: {np.max(answer_lengths)}, "
              f"Min: {np.min(answer_lengths)}")
        
        # Question statistics
        questions = split_data['question']
        question_lengths = [len(q.split()) for q in questions]
        print(f"  Question length - Mean: {np.mean(question_lengths):.2f}, "
              f"Median: {np.median(question_lengths):.2f}, "
              f"Max: {np.max(question_lengths)}, "
              f"Min: {np.min(question_lengths)}")
        
        # Most common answers (for closed-ended)
        if closed_count > 0:
            closed_answers = [answers[i] for i, qt in enumerate(q_types) if qt == 'closed-ended']
            answer_freq = Counter(closed_answers)
            print(f"  Top 5 answers (closed-ended): {answer_freq.most_common(5)}")

print("\n" + "="*60)
print("Preprocessing completed successfully!")
print("="*60)